# 06 · Modelado: Bi-NOA-LSTM multi-horizonte directo

| | |
|---|---|
| **Fase CRISP-DM** | 4 — Modelado |
| **Objetivo** | Entrenar la red Bi-NOA-LSTM (y la ablación Bi-LSTM estándar) que, a partir de 30 días de una celda, predice en un solo pase los 180 días siguientes. |
| **Entradas** | `data/gold/series_filtradas.parquet` (solo train), `data/gold/escaladores.json` |
| **Salidas** | `models/bi_noa_lstm.pt`, `models/bi_lstm_baseline.pt`, `reports/modelado/` (curvas e historial) |

**Esquema de canal independiente con pesos compartidos**
- Una muestra = (celda, origen `t0`). Entrada `[B, 30, 4]`: voltaje filtrado relativo al ancla ÷ σ_Δ, `kA_z`, `vcel_z` y canal-máscara (día con calidad válida).
- Identidad: `nn.Embedding(362, 8)` + indicador de electrolizador. Salida `[B, 180]`: `ŷ_h = (V(t0 + h) − ancla) / σ_Δ`.
- Se excluyen los orígenes cuya ventana toca un tramo `fuera_servicio`; los objetivos sin calidad válida se excluyen del loss (**MSE enmascarado**).

**Celda NOA:** LSTM estándar salvo la salida del estado oculto, `h_t = o_t ⊙ c_t` (sin `tanh`); bidireccional, con el recorrido temporal en `torch.jit.script`. Estabilidad: *gradient clipping* (norma 1,0) y dropout.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import json
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import model as mdl

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "modelado"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"PyTorch {torch.__version__} | USE_NOA={config.USE_NOA} | MASK_POST_JUMP={config.MASK_POST_JUMP} | "
      f"REFIT_FULL_TRAIN={config.REFIT_FULL_TRAIN}")
print("Configuración:", config.TRAIN)


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Datos de train (Gold)

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
esc = io.read_json("gold", "escaladores.json")
train = gold[gold["split"] == "train"]
CELLS = sorted(train["celda"].astype(str).unique())
SIGMA = esc["sigma_delta"]
ESC_V = SIGMA  # unidades normalizadas → V

data = ds.SeriesTensors(train, CELLS, SIGMA, DEVICE)
T_ = len(data.dates)
print(f"{len(CELLS)} celdas × {T_} días de train ({data.dates[0].date()} → {data.dates[-1].date()}) | "
      f"σ_Δ = {SIGMA:.4f} V | orígenes usables: {int(data.ok.sum()):,}")

### Bloque de early stopping

El early stopping se hace sobre el **último bloque de train** (`ES_BLOCK_DAYS` días), nunca sobre validación:
- **Train interno:** orígenes anteriores al bloque; sus objetivos se cortan antes del bloque (sin solapamiento).
- **Bloque interno:** orígenes dentro del bloque cada 7 días; objetivos hasta el final de train.

Con la época elegida, el modelo final se **reentrena con todo train** (`REFIT_FULL_TRAIN`), para aprovechar el período más reciente.

In [ ]:
CUT = T_ - config.ES_BLOCK_DAYS
TR_RANGE = (config.LOOKBACK_DAYS - 1, CUT - 1)
VAL_PAIRS = data.origins(CUT, T_ - 2, stride=7)
print(f"Train interno: orígenes hasta {data.dates[CUT - 1].date()} (objetivos < {data.dates[CUT].date()}) | "
      f"≈ {int(data.origins(*TR_RANGE)[0].numel()) // config.TRAIN['stride']:,} muestras por época (stride {config.TRAIN['stride']})")
print(f"Bloque interno: {data.dates[CUT].date()} → {data.dates[-1].date()} | {len(VAL_PAIRS[0]):,} pares (celda, origen)")

## 2. Saltos de nivel en el objetivo

El notebook 03 mostró intervenciones cada ~6 meses que bajan el voltaje de lotes de celdas 0,2–0,7 V de un día para otro, sin aviso en los datos. Si esos saltos quedan dentro del objetivo, la red intenta memorizar *cuándo* ocurrieron en train (la celda y las covariables del período lo permiten), y sobreajusta.

Con `MASK_POST_JUMP`, para cada origen se enmascaran en el loss los objetivos **a partir del próximo salto de nivel** de la celda. Los saltos se detectan con la misma regla de Silver (R9), pero **solo con datos de train** (etiquetas históricas, no entradas del modelo). El modelo aprende así la **trayectoria de degradación sin intervenciones**, que es lo predecible y lo que necesita el mantenimiento para anticipar qué celdas superarán el umbral. En la evaluación se mide contra **todos** los objetivos válidos.

In [ ]:
V_tr = ds.pivot(train, "voltaje", CELLS)
Q_tr = ds.pivot(train, "calidad", CELLS).apply(lambda s: pd.Categorical(s, categories=cl.CALIDAD).codes)
saltos_tr = cl.level_shifts(V_tr, Q_tr)
data.set_level_shifts(saltos_tr)
x, y, m_all, *_ = data.batch(*data.origins(*TR_RANGE, stride=5), CUT - 1)
_, _, m_jump, *_ = data.batch(*data.origins(*TR_RANGE, stride=5), CUT - 1, mask_jumps=True)
print(f"Saltos en train: {len(saltos_tr)} en {saltos_tr['celda'].nunique()} celdas | "
      f"objetivos conservados con la máscara: {m_jump.sum() / m_all.sum():.1%}")
del x, y, m_all, m_jump

In [ ]:
# Justificación: la ablación rápida con Bi-LSTM estándar, con y sin la máscara de saltos
def entrenar(use_noa, mask_jumps, cfg=config.TRAIN, log=None):
    torch.manual_seed(config.SEED)
    m = mdl.build_model(len(CELLS), len(ds.FEATURES), cfg, use_noa).to(DEVICE)
    r = mdl.fit(m, data, TR_RANGE, CUT - 1, cfg, val_pairs=VAL_PAIRS, val_target_limit=T_ - 1,
                mask_jumps=mask_jumps, log=log)
    return m, r


def tabla_bloque(modelos: dict) -> pd.DataFrame:
    filas = []
    for nombre, m in modelos.items():
        rm, rp = mdl.rmse_by_horizon(m, data, *VAL_PAIRS, T_ - 1)
        filas.append({"modelo": nombre, **{f"RMSE h{h} (V)": float(rm[h - 1]) * ESC_V for h in (1, 30, 60, 180)},
                      "RMSE medio 1–180 (V)": float(rm.mean()) * ESC_V})
    filas.append({"modelo": "persistencia (ancla)", **{f"RMSE h{h} (V)": float(rp[h - 1]) * ESC_V for h in (1, 30, 60, 180)},
                  "RMSE medio 1–180 (V)": float(rp.mean()) * ESC_V})
    return pd.DataFrame(filas).set_index("modelo").round(4)


t = time.time()
m_sin, r_sin = entrenar(False, False)
m_con, r_con = entrenar(False, True)
print(f"Ablación de la máscara en {time.time() - t:.0f} s | mejor época: sin máscara {r_sin['mejor_epoca']}, "
      f"con máscara {r_con['mejor_epoca']}")
tabla_bloque({"Bi-LSTM sin máscara de saltos": m_sin, "Bi-LSTM con máscara de saltos": m_con})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for nombre, r in [("sin máscara", r_sin), ("con máscara", r_con)]:
    h = pd.DataFrame(r["historia"])
    l, = ax.plot(h["epoca"], h["train_mse"], label=f"train · {nombre}")
    ax.plot(h["epoca"], h["val_mse"], ls="--", color=l.get_color(), label=f"bloque interno · {nombre}")
ax.set(yscale="log", xlabel="época", ylabel="MSE (normalizado)", title="Bi-LSTM estándar: efecto de la máscara de saltos")
ax.legend(fontsize=8); fig.tight_layout(); savefig(fig, "01_mascara_saltos")
del m_sin, m_con

**Lectura.**
- **Sin la máscara** de saltos, el mejor punto llega en la época 2: después, la pérdida de train sigue bajando, pero la del bloque interno sube. La red sobreajusta a los saltos de train y termina **peor que la persistencia** (RMSE medio de 0,0255 frente a 0,0209 V).
- **Con la máscara**, entrena de forma estable durante decenas de épocas y **supera a la persistencia** en todos los cortes (0,0173 frente a 0,0209 V de media; 0,0294 frente a 0,0348 V a 180 días).
- La máscara conserva el ~94 % de los objetivos de train.

**Advertencia.** El bloque interno (ago-2024 → abr-2025) casi no tiene intervenciones, así que favorece a un modelo que proyecta la trayectoria sin saltos. La prueba real es la validación del notebook 07, que tiene ~80 saltos: allí las métricas se separan entre pronósticos que cruzan y que no cruzan un salto.

## 3. Entrenamiento de Bi-NOA-LSTM y de la ablación Bi-LSTM estándar

In [ ]:
resultados = {}
for nombre, use_noa in [("Bi-NOA-LSTM", True), ("Bi-LSTM estándar", False)]:
    print(f"\n=== {nombre}: early stopping sobre el bloque interno ===")
    t = time.time()
    m_es, r_es = entrenar(use_noa, config.MASK_POST_JUMP, log=print)
    t_es = time.time() - t
    print(f"→ mejor época {r_es['mejor_epoca']} ({t_es:.0f} s, {mdl.count_params(m_es):,} parámetros)")
    final, r_fin, t_fin = m_es, None, 0.0
    if config.REFIT_FULL_TRAIN:
        print(f"=== {nombre}: reentrenamiento con todo train ({r_es['mejor_epoca']} épocas) ===")
        t = time.time()
        torch.manual_seed(config.SEED)
        final = mdl.build_model(len(CELLS), len(ds.FEATURES), config.TRAIN, use_noa).to(DEVICE)
        r_fin = mdl.fit(final, data, (config.LOOKBACK_DAYS - 1, T_ - 2), T_ - 1, config.TRAIN,
                        epochs=r_es["mejor_epoca"], mask_jumps=config.MASK_POST_JUMP, log=None)
        t_fin = time.time() - t
        print(f"→ {t_fin:.0f} s")
    resultados[nombre] = {"use_noa": use_noa, "es": m_es, "r_es": r_es, "final": final, "r_fin": r_fin,
                          "segundos": t_es + t_fin}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for nombre, r in resultados.items():
    h = pd.DataFrame(r["r_es"]["historia"])
    l, = axes[0].plot(h["epoca"], h["train_mse"], label=f"{nombre} · train")
    axes[0].plot(h["epoca"], h["val_mse"], ls="--", color=l.get_color(), label=f"{nombre} · bloque interno")
    axes[0].axvline(r["r_es"]["mejor_epoca"], color=l.get_color(), ls=":", lw=0.8)
    rm, rp = mdl.rmse_by_horizon(r["es"], data, *VAL_PAIRS, T_ - 1)
    axes[1].plot(range(1, config.HORIZON_DAYS + 1), rm.cpu() * ESC_V, color=l.get_color(), label=nombre)
axes[1].plot(range(1, config.HORIZON_DAYS + 1), rp.cpu() * ESC_V, color="k", ls="--", label="persistencia (ancla)")
axes[0].set(yscale="log", xlabel="época", ylabel="MSE (normalizado)", title="Curvas de entrenamiento (early stopping)")
axes[1].set(xlabel="horizonte (días)", ylabel="RMSE (V)", title="Error vs. horizonte en el bloque interno de train")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.tight_layout(); savefig(fig, "02_curvas_entrenamiento")

bloque = tabla_bloque({n: r["es"] for n, r in resultados.items()})
bloque["mejor época"] = [r["r_es"]["mejor_epoca"] for r in resultados.values()] + [None]
bloque["tiempo (s)"] = [round(r["segundos"]) for r in resultados.values()] + [None]
bloque

**Lectura.**
- En el bloque interno, **Bi-NOA-LSTM y Bi-LSTM estándar quedan prácticamente empatados**. Los dos superan a la persistencia a 60 días (0,0214 y 0,0216 frente a 0,0223 V, una mejora pequeña) y con más margen a 180 días (0,0297 y 0,0294 frente a 0,0348 V).
- La NOA no aporta ventaja: es algo peor a 1 día (0,0045 frente a 0,0031 V) y cuesta ~5 veces más tiempo de entrenamiento (su bucle en TorchScript no usa cuDNN). Esa comparación es justamente la ablación que pedía la especificación.
- **Error vs. horizonte:** el RMSE crece con el horizonte en todos los modelos, como se esperaba. Los picos con período de 7 días no son del modelo: los orígenes están espaciados 7 días, así que unas pocas fechas objetivo anómalas del bloque caen siempre en los mismos horizontes. Son 2024-09-07, 2024-10-06 y los tres últimos días de train (26–28 abr 2025), con desviaciones de 0,05–0,075 V, y afectan por igual a los modelos y a la persistencia.
- Tras elegir la época, los dos modelos se reentrenaron con todo train (20 y 36 épocas). Los checkpoints guardados corresponden a esos modelos finales, así que las métricas de esta tabla (de los modelos del early stopping) son solo orientativas.

## 4. Guardado de los modelos

In [ ]:
vmd_params = json.loads((config.MODELS_DIR / "vmd_params.json").read_text(encoding="utf-8"))
split = gold.loc[gold["split"] == "val", "fecha"].min()
archivos = {"Bi-NOA-LSTM": "bi_noa_lstm.pt", "Bi-LSTM estándar": "bi_lstm_baseline.pt"}
historial = []
for nombre, r in resultados.items():
    meta = {
        "nombre": nombre, "use_noa": r["use_noa"], "cells": CELLS, "features": ds.FEATURES,
        "train_cfg": config.TRAIN, "lookback": config.LOOKBACK_DAYS, "horizon": config.HORIZON_DAYS,
        "anchor_days": config.ANCHOR_DAYS, "escaladores": esc, "vmd_params": vmd_params,
        "mask_post_jump": config.MASK_POST_JUMP, "refit_full_train": config.REFIT_FULL_TRAIN,
        "mejor_epoca": r["r_es"]["mejor_epoca"], "entrenado_hasta": str(data.dates[-1].date()),
        "es_block_desde": str(data.dates[CUT].date()), "metricas_bloque_interno": bloque.loc[nombre].to_dict(),
        "fecha_entrenamiento": pd.Timestamp.now().isoformat(timespec="seconds"), "seed": config.SEED,
    }
    path = config.MODELS_DIR / archivos[nombre]
    mdl.save_checkpoint(path, r["final"], meta)
    print(f"✔ {path.relative_to(config.BASE_DIR)} ({path.stat().st_size / 1e6:.2f} MB)")
    historial += [{"modelo": nombre, "fase": "early_stopping", **h} for h in r["r_es"]["historia"]]
    if r["r_fin"]:
        historial += [{"modelo": nombre, "fase": "reentrenamiento", **h} for h in r["r_fin"]["historia"]]
pd.DataFrame(historial).to_csv(FIG_DIR / "historial_entrenamiento.csv", index=False)
bloque.to_csv(FIG_DIR / "metricas_bloque_interno.csv")
print(f"✔ {(FIG_DIR / 'historial_entrenamiento.csv').relative_to(config.BASE_DIR)}")

## 5. Verificaciones

In [ ]:
x, y, mask, cell, el = data.batch(*VAL_PAIRS, T_ - 1)
checks = {}
for nombre, archivo in archivos.items():
    m, ck = mdl.load_checkpoint(config.MODELS_DIR / archivo, DEVICE)
    with torch.no_grad():
        p1 = m(x[:512], cell[:512], el[:512])
        p2 = resultados[nombre]["final"].eval()(x[:512], cell[:512], el[:512])
    checks[f"{nombre}: el checkpoint reproduce las predicciones"] = torch.allclose(p1, p2, atol=1e-5)
    checks[f"{nombre}: salida [B, {config.HORIZON_DAYS}] finita"] = p1.shape[1] == config.HORIZON_DAYS and bool(torch.isfinite(p1).all())
checks["solo se usaron fechas de train"] = data.dates[-1] < split
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values())

## Conclusión

- Se entrenaron **Bi-NOA-LSTM** (celda propia con `h_t = o_t ⊙ c_t` en TorchScript) y la ablación **Bi-LSTM estándar**, con pesos compartidos entre las 362 celdas y salida directa de 180 días.
- La decisión clave fue **enmascarar los objetivos posteriores a un salto de nivel**. Sin ella, la red sobreajusta a las intervenciones de train y no supera a la persistencia.
- En el bloque interno de train, los dos modelos superan a la persistencia, sobre todo a 180 días, y la celda NOA no mejora al LSTM estándar.
- Salidas: `models/bi_noa_lstm.pt` y `models/bi_lstm_baseline.pt`, con pesos, configuración, escaladores, parámetros de VMD, celdas y versión. Las curvas y el historial están en `reports/modelado/`.

**Siguiente:** notebook 07 — Evaluación rolling-origin sobre validación: modelos frente a persistencia y tendencia lineal, error vs. horizonte, y métricas separadas por pronósticos que cruzan o no un salto.